# IA Local con Ollama

> Aprende a ejecutar modelos de lenguaje completamente en local usando Ollama,
> sin enviar datos a ninguna API externa. Ideal para privacidad y trabajo offline.

## Objetivos
- Verificar la instalación de Ollama y listar modelos disponibles
- Realizar llamadas básicas con el SDK de Ollama
- Implementar streaming de respuestas en tiempo real
- Construir un chat conversacional con historial

**Requisito previo:** Ollama instalado (`https://ollama.ai`) y al menos un modelo descargado:
```bash
ollama pull llama3.2
```

## 1. Instalación de dependencias

In [ ]:
%pip install ollama requests --quiet

## 2. Verificar que Ollama está disponible

In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434"
MODELO = "llama3.2"  # Cambia por el modelo que tengas descargado

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    if respuesta.status_code == 200:
        datos = respuesta.json()
        modelos = [m["name"] for m in datos.get("models", [])]
        print(f"✓ Ollama disponible en {OLLAMA_URL}")
        print(f"Modelos instalados: {', '.join(modelos) if modelos else 'Ninguno'}")
        if modelos:
            MODELO = modelos[0]  # Usar el primer modelo disponible
            print(f"Usando modelo: {MODELO}")
    else:
        print(f"✗ Ollama respondió con código {respuesta.status_code}")
except requests.ConnectionError:
    print("✗ Ollama no está disponible. Asegúrate de que está instalado y ejecutando.")
    print("  Instala desde: https://ollama.ai")
    print("  Inicia con: ollama serve")

## 3. Llamada básica con el SDK de Ollama

In [ ]:
import ollama

# Llamada simple — equivalente a una llamada de API pero 100% local
respuesta = ollama.chat(
    model=MODELO,
    messages=[{"role": "user", "content": "Explica qué es la inteligencia artificial en 3 frases."}]
)

print("=== RESPUESTA DEL MODELO LOCAL ===")
print(f"Modelo: {respuesta.model}")
print(f"Respuesta:\n{respuesta.message.content}")
print(f"\nTokens generados: {respuesta.eval_count}")
print(f"Duración total: {respuesta.total_duration / 1e9:.2f} segundos")

## 4. Streaming de respuestas en tiempo real

In [ ]:
import sys

print("=== RESPUESTA CON STREAMING ===")
print("Generando... ", end="", flush=True)
print()

# stream=True devuelve un generador que emite tokens en tiempo real
for chunk in ollama.chat(
    model=MODELO,
    messages=[{"role": "user", "content": "Lista 5 ventajas de usar modelos de IA en local."}],
    stream=True
):
    texto = chunk.message.content
    print(texto, end="", flush=True)

print()  # Salto de línea al terminar

## 5. Chat conversacional con historial

In [ ]:
def chat_local(historial: list[dict], pregunta: str) -> tuple[str, list[dict]]:
    """Envía una pregunta manteniendo el historial de conversación."""
    historial.append({"role": "user", "content": pregunta})

    respuesta = ollama.chat(model=MODELO, messages=historial)
    respuesta_texto = respuesta.message.content

    historial.append({"role": "assistant", "content": respuesta_texto})
    return respuesta_texto, historial

# Conversación de ejemplo con contexto
historial = [{"role": "system", "content": "Eres un asistente experto en Python. Responde de forma concisa."}]

conversacion = [
    "¿Cuál es la diferencia entre una lista y una tupla en Python?",
    "¿En qué caso usarías una sobre la otra?",
    "Dame un ejemplo de código con cada una."
]

for pregunta in conversacion:
    print(f"\nUsuario: {pregunta}")
    respuesta, historial = chat_local(historial, pregunta)
    print(f"Asistente: {respuesta[:300]}{'...' if len(respuesta) > 300 else ''}")

print(f"\nTurnos en la conversación: {len([m for m in historial if m['role'] == 'user'])}")

## 6. Llamada via API REST (sin SDK)

In [ ]:
# Ollama expone una API REST — útil para integrar desde cualquier lenguaje
payload = {
    "model": MODELO,
    "messages": [{"role": "user", "content": "¿Qué es el aprendizaje por refuerzo?"}],
    "stream": False
}

respuesta = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=60)
datos = respuesta.json()

print("=== LLAMADA DIRECTA A LA API REST ===")
print(f"Status: {respuesta.status_code}")
print(f"Respuesta: {datos['message']['content'][:400]}")
print(f"\nTokens/segundo: {datos.get('eval_count', 0) / (datos.get('eval_duration', 1) / 1e9):.1f}")